# PyGGUF: GitHub-backed Kaggle server

This notebook clones `https://github.com/binief/pytorch-engine` and runs its Python engine. First push the accompanying source bundle to the repository's `main` branch. Each fresh Kaggle session needs repository access and an installed CUDA-enabled PyTorch / Triton runtime.

**Security:** the default runs the latest code from the remote main branch. Review the repository, or set `GIT_REF` below to a trusted full commit SHA for a reproducible run.


In [ ]:
# Install lightweight web/hub dependencies. PyTorch and Triton are expected to be preinstalled on Kaggle.
!pip install -q fastapi uvicorn openai gguf "huggingface_hub[hf_transfer]" tokenizers jinja2


In [ ]:
# Clone or update the engine repository. Stop an existing server before reloading its code.
import os, subprocess, sys
from pathlib import Path

GIT_URL = "https://github.com/binief/pytorch-engine.git"
GIT_REF = "main"  # Replace with a reviewed full commit SHA for reproducible runs.
ENGINE_DIR = Path("/kaggle/working/pytorch-engine") if Path("/kaggle").exists() else Path.cwd() / "pytorch-engine"

if globals().get("server_handle") is not None:
    server_handle.stop(timeout=60.0)
    server_handle = None
if not ENGINE_DIR.exists():
    subprocess.run(["git", "clone", GIT_URL, str(ENGINE_DIR)], check=True)
elif not (ENGINE_DIR / ".git").exists():
    raise RuntimeError(f"{ENGINE_DIR} exists but is not a Git checkout; choose a different directory.")
else:
    remote = subprocess.check_output(["git", "-C", str(ENGINE_DIR), "remote", "get-url", "origin"], text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != GIT_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Unexpected repository origin: {remote}")
    dirty = subprocess.check_output(["git", "-C", str(ENGINE_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError(f"Checkout {ENGINE_DIR} has local changes; save or remove them before updating.")
    subprocess.run(["git", "-C", str(ENGINE_DIR), "fetch", "--prune", "origin"], check=True)

# Resolve a branch on origin, or a specific commit already obtained by clone/fetch.
target = "origin/main" if GIT_REF == "main" else GIT_REF
subprocess.run(["git", "-C", str(ENGINE_DIR), "checkout", "--detach", target], check=True)
commit = subprocess.check_output(["git", "-C", str(ENGINE_DIR), "rev-parse", "HEAD"], text=True).strip()
for module_name in list(sys.modules):
    if module_name == "kaggle_serve" or module_name == "pygguf_engine" or module_name.startswith("pygguf_engine."):
        del sys.modules[module_name]
sys.path.insert(0, str(ENGINE_DIR))
print(f"PyGGUF source: {ENGINE_DIR} @ {commit}")


## 2. Download Any Hybrid (`Qwen3.8-27B` / `Qwen3.5`), Dense, or Sparse MoE `.gguf` into `/kaggle/tmp`
Uses `/kaggle/tmp` (~1.1 TB disk) + `hf_transfer` (1–2 Gbps) so `/kaggle/working`'s 20 GB limit is never touched.

In [ ]:
import sys
from pathlib import Path

from kaggle_serve import download_gguf_kaggle, launch_kaggle_server

# --- Model: Qwen3.8-27B UD-Q4_K_M on dual T4 ---
REPO_ID = "unsloth/Qwen3.8-27B-GGUF"
FILENAME = "Qwen3.8-27B-UD-Q4_K_M.gguf"
TOKENIZER_REPO = None  # None = uses fast Rust tokenizer + hardened Agentic Jinja2 template!

# --- Preset B: Hybrid Gated-DeltaNet + Gated Attention (27B Unsloth Dynamic Q4_K_M with TP=2 on Dual T4!) ---
# REPO_ID = "unsloth/Qwen3.8-27B-GGUF"          # (or point gguf_path directly to /kaggle/tmp/...)
# FILENAME = "Qwen3.8-27B-UD-Q4_K_M.gguf"

# --- Preset C: Sparse MoE (30B-35B quality, only 3B active weights read per token -> highest tok/s!) ---
# REPO_ID = "Qwen/Qwen3-30B-A3B-GGUF"
# FILENAME = "Qwen3-30B-A3B-Q4_K_M.gguf"

gguf_path = download_gguf_kaggle(repo_id=REPO_ID, filename=FILENAME)


## 3. Launch Turbo 256K-Ready OpenAI Server + Adaptive `draft-mtp` + Cloudflare HTTPS Tunnel
- **`kv_dtype="int4"` (Default for Long Context)**: Cuts KV cache VRAM & Flash-Decode bandwidth by **~75% vs FP16 / ~50% vs INT8**, unlocking **128K–256K+ context** on Dual T4 (`"int8"` and `"fp16"` also supported).
- **`speculative_mtp=True` + `mtp_adaptive=True` (`k = 1..3`) + `mtp_kv_window=4096`**: Loads embedded GGUF `blk.{n_layers}.nextn.*` with a bounded `4096`-token ring KV cache (`~4 MiB` VRAM even at 256K!) and chains up to `3` draft tokens when GPU softmax confidence `>= 0.55`. Set **`speculative_mtp=False`** to skip loading `blk.{n_layers}` (**saves ~0.5 GB VRAM**), or toggle live per request via `extra_body={"speculative_mtp": False}`!
- **RoPE context extension**: checkpoints trained for a shorter window can be pushed to `max_context_len` with `rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=32768` (YaRN mscale attention factor applied automatically; `linear` and `ntk-aware` also supported). GGUF `{arch}.rope.scaling.*` metadata is used automatically when present. Extension never re-trains weights — quality is checkpoint-dependent.
- **`parallel_mode="auto"`**: Automatically enables **2-GPU Tensor Parallelism (`TP=2`, 640 GB/s combined memory bandwidth)** on Dual T4 for Dense & Hybrid (`Qwen3.8-27B`) models, or **Pipeline Parallelism (`PP`)** for Sparse MoE models.
- **`enable_thinking=True`**: Extracts `<think>...</think>` into `reasoning_content` (**zero `<think></think>` tags in `content`!**). Set **`enable_thinking=False`** for **3x–5x faster end-to-end replies** when chain-of-thought isn't needed.

In [ ]:
import gc
import torch

# Cleanly stop any previous server_handle if re-running this cell in the same Kaggle session
if "server_handle" in globals() and server_handle is not None:
    # Wait for active requests to finish; do not launch a second model on timeout.
    server_handle.stop(timeout=60.0)
    server_handle = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

API_KEY = "kaggle-pygguf-key"   # Set to None to disable Bearer auth
ENABLE_PUBLIC_TUNNEL = True      # Exposes https://xxxx.trycloudflare.com/v1 for OpenCode / Claude Code / Cline / Cursor

server_handle = launch_kaggle_server(
    gguf_path=gguf_path,
    hf_tokenizer=TOKENIZER_REPO,
    chat_template=None,          # Optional path to a custom .jinja file (None = built-in hardened template)
    chat_template_mode="auto",   # "auto" | "qwen35_xml" | "qwen3_json" | "gguf"
    enable_thinking=True,        # Set False for 3x-5x faster turn completion without <think> CoT!
    include_reasoning=True,      # Set False to completely omit reasoning_content from JSON/SSE responses
    cpu_threads=4,              # PyTorch CPU intra-op threads; benchmark 2, 4 and 8
    parallel_mode="auto",        # "auto" = TP=2 (640 GB/s) on Dual-T4 Dense/Hybrid, PP on MoE | "tp" | "pp"
    enable_cuda_graph=True,      # Captures M=1 CUDA Graphs automatically on PP / single-GPU runs
    speculative_mtp=True,        # Native GGUF `blk.{L}.nextn.*` draft-mtp head (set False to save ~0.5 GB VRAM!)
    mtp_draft_tokens=None,       # None = Confidence-Gated Adaptive k=1..3 (or pass 1..8 to pin fixed k)
    mtp_adaptive=True,           # Chains k=2..3 MTP tokens when cumulative softmax confidence >= 0.55!
    mtp_kv_window=4096,          # Bounded 4096-token Ring KV for MTP (~4 MiB VRAM even at 256K trunk context!)
    speculative_ngram=True,      # Hybrid zero-VRAM N-gram suffix drafter + draft-mtp across ALL temperatures!
    max_context_len=131072,      # 128K context out of the box (set 262144 for full 256K — ~2.1 GiB KV in int4!)
    kv_dtype="int4",             # "int4" (~75% VRAM & BW savings, 256K-ready!) | "int8" | "fp16"
    # RoPE context extension (only needed when the checkpoint's TRAINED ctx is shorter than max_context_len):
    # GGUF {arch}.rope.scaling.* metadata is honored automatically (llama.cpp-compatible keys), otherwise override:
    # rope_scaling_type="yarn", rope_scaling_factor=4.0, rope_scaling_original_context_length=32768,
    port=8000,
    api_key=API_KEY,
    enable_tunnel=ENABLE_PUBLIC_TUNNEL,
    background=True,
)


## 4. Test Streaming Speed, Clean `reasoning_content`, & Native Multi-Turn Tool Calling

In [ ]:
import json
import time
from openai import OpenAI

client = OpenAI(
    base_url=server_handle.local_base_url,  # Use server_handle.public_base_url in external harnesses!
    api_key=API_KEY,
)

print("Models:", [m.id for m in client.models.list().data])
print("VRAM, Parallelism & MTP Report:", server_handle.engine.vram_report())

# --- Test A: Streaming Chat (notice zero <think></think> in delta.content!) ---
print("\n=== 1. Streaming Chat (TP=2 + Native draft-mtp) ===")
t0 = time.perf_counter()
stream = client.chat.completions.create(
    model=server_handle.engine.model_id,
    messages=[
        {"role": "developer", "content": "You are a concise AI systems assistant."},
        {"role": "user", "content": "Give 2 short bullet points on why TP=2 + draft-mtp accelerates Dual T4 GPUs."},
    ],
    temperature=0.6,
    max_tokens=256,
    stream=True,
    stream_options={"include_usage": True},
    # Optional live per-request overrides:
    # extra_body={"speculative_mtp": False}                  # disable MTP for this call only
    # extra_body={"speculative_mtp": True, "mtp_draft_tokens": 2}
)
printed_think_header = False
printed_ans_header = False
comp_toks = 0
for chunk in stream:
    if getattr(chunk, "usage", None) is not None:
        comp_toks = chunk.usage.completion_tokens
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    rc = getattr(delta, "reasoning_content", None)
    if rc:
        if not printed_think_header:
            print("[Reasoning]: ", end="", flush=True)
            printed_think_header = True
        print(rc, end="", flush=True)
    if delta.content:
        if not printed_ans_header:
            print("\n\n[Answer]:\n", end="", flush=True)
            printed_ans_header = True
        print(delta.content, end="", flush=True)
elapsed = time.perf_counter() - t0
print(f"\n(Generated {comp_toks} tokens in {elapsed:.2f}s -> {comp_toks / max(elapsed, 1e-6):.1f} e2e tok/s)")

# --- Test B: Native OpenAI Multi-Turn Tool Calling ---
print("\n=== 2. Native OpenAI Tool Calling ===")
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_gpu_cluster_status",
            "description": "Query live GPU VRAM and temperature metrics for a cluster node.",
            "parameters": {
                "type": "object",
                "properties": {
                    "node_name": {"type": "string", "description": "Cluster node identifier"},
                    "include_temps": {"type": "boolean", "description": "Include Celsius temperatures"},
                },
                "required": ["node_name"],
            },
        },
    }
]

tool_msgs = [{"role": "user", "content": "Check GPU cluster status on node kaggle-t4x2 including temperatures."}]
r1 = client.chat.completions.create(
    model=server_handle.engine.model_id,
    messages=tool_msgs,
    tools=tools,
    temperature=0.0,
    max_tokens=384,
)
msg1 = r1.choices[0].message
print("Finish Reason:", r1.choices[0].finish_reason, "| Timings (incl. MTP):", r1.model_extra.get("timings"))
print("Parsed Tool Calls:", msg1.tool_calls)

if msg1.tool_calls:
    tc = msg1.tool_calls[0]
    args = json.loads(tc.function.arguments)
    tool_msgs.append(msg1.model_dump(exclude_none=True))
    tool_msgs.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps({"node": args.get("node_name"), "gpus": "2x Tesla T4", "free_vram_gib": 8.4, "temp_c": 58}),
        }
    )
    r2 = client.chat.completions.create(
        model=server_handle.engine.model_id,
        messages=tool_msgs,
        tools=tools,
        temperature=0.2,
        max_tokens=256,
    )
    print("Final Tool-Grounded Answer:", r2.choices[0].message.content)


In [ ]:
# Optional: Keep cell alive for hours if using external harnesses (OpenCode / Claude Code / Cline / Continue / Cursor)
# import time
# while True:
#     time.sleep(60)
